# ANOVA 
## Introduction


## Mathematical concepts in one-way ANOVA
### Variance


### Partitioning the sum of squares


### F-statistic


### P value


### Effect size
#### Eta-squared
#### Omega-squared


## Calculating one-way ANOVA manually
### Exploratory data analysis


In [ ]:
import pandas as pd
import pingouin as pg

# Load the 'anova' dataset from pingouin
# List more dataset using pg.list_dataset()
data_anova = pg.read_dataset('anova')

print("First rows of the hair color 'anova' dataframe:")
print(data_anova.head())

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(3.5, 3))

# Create the boxplot
sns.boxplot(
    x='Hair color',
    y='Pain threshold',
    data=data_anova,
    hue='Hair color',
    showfliers=False)

# Overlay the stripplot
sns.stripplot(
    x='Hair color', y='Pain threshold', data=data_anova,
    color='black', size=7)

# Add plot labels and title
plt.xticks(rotation=45, ha='right', rotation_mode='anchor')
plt.title('Pain threshold vs. hair color');

### Getting the sums of squares


In [ ]:
# Parameters of the analysis
r = data_anova['Hair color'].nunique() # Number of groups/conditions
N = len(data_anova)                    # Total number of data points
DF_between = r - 1                     # DF for MSeffect
DF_within  = N - r                     # DF for MSerror

print(f"There are {r:n} different conditions and {N:n} total data points")
print(
    f"This leads to DF={DF_between:n} for the effect (SSB), \
and DF={DF_within:n} for error (SSW) variance")

In [ ]:
# Calculate the overall mean of 'Pain threshold'
grand_mean = data_anova['Pain threshold'].mean()
print(f"Grand mean = {grand_mean:.2f}")

# Sums of squares
# $\text{SST} = \sum_i \sum_j (x_{ij} - \bar x)^2$
SST = (
    (data_anova['Pain threshold'] - grand_mean)**2
).sum()

# $\text{SSW} = \sum_i \sum_j (x_{ij} - \bar x_i)^2$
SSW = (
    data_anova.groupby('Hair color')['Pain threshold']
    .transform(lambda x: (x - x.mean())**2)
).sum()


# Compute SSB from scratch
# \text{SSB} = \sum_i n_i (\bar{x_i} - \bar{x})^2$
SSB = (
    data_anova.groupby('Hair color')['Pain threshold']
    .apply(lambda x: x.count() * (x.mean() - grand_mean)**2)
).sum()

# or $\text{SST} = \text{SSB} + \text{SSW}$
print()
print("Sums of squares calculated manually:")
print(" SST =", round(SST, 1))
print(" SSW =", round(SSW, 1))
print(" SSB =", round(SSB, 1))

### ANOVA reconstruction from summary data


#### Summary statistics


In [ ]:
group_statistics = (
    data_anova.groupby('Hair color')
    ['Pain threshold']
    .agg(['mean', 'std', 'count'])
)

print("Summary group statistics for 'Pain threshold':")
print(group_statistics.round(3))

#### Calculating SSB


In [ ]:
def sum_squares_between(n, group_mean, grand_mean):
    """
    Returns the sum of squares between (SSB) for a group inside an `apply` 
    call, based on the formula: $\\sum_i(n_i (\bar{x}_i - \bar{x})^2)$
    """

    return n * (group_mean - grand_mean)**2

In [ ]:
ssb_individual = (
    group_statistics.apply(
        lambda row: sum_squares_between(
            n=row['count'],
            group_mean=row['mean'],
            grand_mean=grand_mean),
        axis=1))

print("Group SSBs calculated from group means and sizes, \
and the grand mean:")
print(ssb_individual.round(2))
print()
print("Total SSB =", round(ssb_individual.sum(), 3))

#### Calculating SSW


In [ ]:
def sum_squares_within(n, group_std):
    """
    Returns the sum of squares within (SSW) for a group inside an `apply` 
    call, based on the formula: $\\sum_i((n_i - 1) s_i^2)$
    """

    return (n - 1) * group_std**2

In [ ]:
ssw_individual = (
    group_statistics.apply(
        lambda row: sum_squares_within(
            n=row['count'],
            group_std=row['std']),
        axis=1)
)

print("Group SSW calculated from groups standard deviations and sizes:")
print(ssw_individual.round(2))
print()
print("Total SSW =", round(ssw_individual.sum(), 3))

### Calculating F-ratio and P value


In [ ]:
# Calculate the MS
MS_between = SSB / DF_between
MS_within  = SSW / DF_within

print(f"MS_between = {MS_between:.1f}")
print(f"MS_within = {MS_within:.1f}")

In [ ]:
from scipy.stats import f as f_dist

# F ratio and associated P value
f_ratio = MS_between / MS_within
p_value = f_dist(dfn=DF_between, dfd=DF_within).sf(f_ratio)

print(f"With an F-ratio = {f_ratio:.3f}, the associated P value = \
{p_value:.5f}")

### Visualizing the F-distribution and critical values


In [ ]:

import numpy as np

# Significance level (alpha)
α = 0.05

# Calculate critical F-value
f_crit = f_dist(dfn=DF_between, dfd=DF_within).ppf(1 - α)

# Generate x values for plotting
x_f = np.linspace(0, 10, 500)  # Adjusted range for better visualization
hx_f = f_dist(DF_between, DF_within).pdf(x_f)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x_f, hx_f, lw=2, color='black')

# Critical value
plt.axvline(
    x=f_crit,
    color='orangered', linestyle='--',
    label=f"F* ({f_crit:.2f})")

# Alpha area
plt.fill_between(
    x_f[x_f >= f_crit], hx_f[x_f >= f_crit],
    color='tomato', label=f"α ({α})")

# F-statistic
plt.axvline(
    x=f_ratio,
    color='limegreen', linestyle='-.',
    label=f"F ({f_ratio:.2f})")

# P value area
plt.fill_between(
    x_f[x_f >= f_ratio], hx_f[x_f >= f_ratio],
    color='greenyellow', label=f"P ({p_value:.3f})")

plt.xlabel("F")
plt.ylabel('Density')
plt.title(f"F-distribution ({DF_between},{DF_within})")
plt.margins(x=0.01, y=0)
plt.legend();

### ANOVA table


## One-way ANOVA with Python
### One-way ANOVA with Pingouin


#### Normality test


In [ ]:
print("Normality test (Shapiro-Wilk):")
print(
    pg.normality(
        data=data_anova,
        dv='Pain threshold',
        group='Hair color'))

In [ ]:

from scipy.stats import probplot

fig, axes = plt.subplots(1, 4, figsize=(6, 1.75))

for i, color in enumerate(data_anova['Hair color'].unique()):
    group_data = data_anova[
        data_anova['Hair color'] == color]['Pain threshold']
    probplot(group_data, plot=axes[i])
    axes[i].set_title(f"{color}", fontsize=10)
    axes[i].set_xlabel('Theoretical quantiles', fontsize=8)
    axes[i].set_ylabel('Sample quantiles', fontsize=8)
plt.tight_layout();

#### Homoscedasticity


In [ ]:
print("Test for equal variance (Levene):")
print(
    pg.homoscedasticity(
        data=data_anova,
        dv='Pain threshold',
        group='Hair color'))

#### Conducting ANOVA


In [ ]:
aov = pg.anova(
    data=data_anova,
    dv='Pain threshold',
    between='Hair color',
    detailed=True)

print("One-way ANOVA (Pinguin):")
print(aov.round(4))

In [ ]:
print("One-way ANOVA (Pingouin; compact output):")
print(
    data_anova.anova(
        dv='Pain threshold',
        between='Hair color',
        detailed=False,  # More compact output
        effsize='n2'))

In [ ]:
omega_squared = (SSB - (r - 1) * MS_within) / (SST + MS_within)

print("ω² =", round(omega_squared, 5))

### One-way ANOVA with SciPy


In [ ]:
from scipy.stats import f_oneway

# Extract array of data for each individual group
hair_colors = data_anova['Hair color'].unique()
groups = [
    data_anova['Pain threshold'][data_anova['Hair color'] == color]
    for color in hair_colors]

# Perform one-way ANOVA using SciPy
F_scipy, p_scipy = f_oneway(*groups)

# similar to:
# F, p = f_oneway(
#     data[data['Hair color'] == 'Dark Blond']['Pain threshold'],
#     data[data['Hair color'] == 'Dark Brunette']['Pain threshold'],
#     data[data['Hair color'] == 'Light Blond']['Pain threshold'],
#     data[data['Hair color'] == 'Light Brunette']['Pain threshold'],
# )

print("One-way ANOVA (SciPy):")
print(f"F statistic = {F_scipy:.3f} with P value = {p_scipy:.5f}")

### One-way ANOVA using statsmodels


In [ ]:
# Suppress all UserWarnings, incl. messages related to small sample size
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from statsmodels.formula.api import ols

formula = "Q('Pain threshold') ~ Q('Hair color')"

# Prepare the one-way ANOVA model using statsmodels
model_anova_statsmodels = ols(formula=formula, data=data_anova)

# Fit the model
results_anova_statsmodels = model_anova_statsmodels.fit()

# Print the model summary
print(results_anova_statsmodels.summary().tables[0])

In [ ]:
from statsmodels.stats.anova import anova_lm

# Obtain the ANOVA table
anova_table_statsmodels = anova_lm(results_anova_statsmodels, typ=1)
print("ANOVA table (statsmodels):")
print(anova_table_statsmodels.round(3))  # Print the rounded ANOVA table

## Repeated measures ANOVA


### Partitioning variance in rmANOVA


### Calculating the F-statistic
### Advantages of rmANOVA


### Calculating rmANOVA manually


In [ ]:
pd.set_option('future.no_silent_downcasting', True)

# Create the DataFrame
data_rmanova = pd.DataFrame({
    'before':
    [165,155,138,150,149,135,145,170,138,144,165,139,141,149,135],
    'during':
    [145, 139, 141, 145, 155, 138, 150, 166, 143, 145, 155, 165, 139,
     141, 137],
    'after':
    [140,133,140,145,149,125,142,160,140,142,133,140,141,140,133]})

# Add the case number as index
data_rmanova.index.name = 'case'

print("First rows of the blood pressure dataset:")
print(data_rmanova.head())

In [ ]:
# Melt the DataFrame into long format after converting 
# the index ('case' numbers) into a regular column
data_rmanova_long = data_rmanova.reset_index().melt(
    id_vars='case',               # Keep 'case' as identifier
    var_name='time',              # Name of the new column for time points
    value_name='blood_pressure',  # Name of the new column for blood pressure
)
print("First rows of the long-format blood pressure dataset:")
print(data_rmanova_long.head())

In [ ]:

plt.figure(figsize=(3.5, 3))

# Create the paired plot
pg.plot_paired(
    data=data_rmanova_long,
    dv='blood_pressure',
    within='time',
    subject='case',
    boxplot=True,
    order=['before', 'during', 'after'],
    # boxplot_in_front=False,  # fix submitted via a PR
    boxplot_kwargs={'color': 'white', 'linewidth': 2, 'zorder': 1})
plt.xlabel('')
plt.ylabel('Blood pressure (mmHg)')
plt.title("Paired plot rmANOVA")
sns.despine(trim=True);  # not default in Pingouin anymore

In [ ]:
# Parameters of the analysis
# Number of repeated measurements
r_rmanova = len(data_rmanova_long['time'].unique())
# Number of subjects
n_rmanova = len(data_rmanova_long['case'].unique())
# Degrees of freedom
DF_time_rmanova = r_rmanova - 1
DF_subjects_rmanova = n_rmanova - 1 
DF_error_rmanova = (n_rmanova - 1) * (r_rmanova - 1)

print(
    f"There are {r_rmanova} repeated measurements and {n_rmanova} subjects")
print(f"This leads to DF={DF_time_rmanova} for SSW, DF=\
{DF_error_rmanova} for SSW, and DF={DF_subjects_rmanova} for SSS")

In [ ]:
# Calculate the overall mean of 'blood_pressure'
grand_mean_rmanova = data_rmanova_long['blood_pressure'].mean()
print(f"Grand mean = {grand_mean_rmanova:.2f}")

# Sums of squares
# SST (total sum of squares)
SST_rmanova = (
    (data_rmanova_long['blood_pressure'] - grand_mean_rmanova)**2).sum()

# SSS (sum of squares for subjects)
SSS_rmanova = (
    data_rmanova_long.groupby('case')['blood_pressure']
  .apply(lambda x: (x.count()) * ((x.mean() - grand_mean_rmanova)**2))
).sum()

# SSB (sum of squares between groups)
SSB_rmanova = (
    data_rmanova_long.groupby(['case', 'time'])['blood_pressure']
    .mean()  # Calculate the mean for each subject-time combination
    .groupby('time')  # Group by time and calculate SSB
    .apply(lambda x: x.count() * (x.mean() - grand_mean_rmanova)**2)
).sum()

# SSW (sum of squares within subjects) using substraction
SSW_rmanova = SST_rmanova - SSS_rmanova - SSB_rmanova

print()
print("Sums of squares calculated manually:")
print(f" SST = {SST_rmanova:.1f}")
print(f" SSW = {SSW_rmanova:.1f}")
print(f" SSB = {SSB_rmanova:.1f}")
print(f" SSS = {SSS_rmanova:.1f}")

In [ ]:
# Calculate MS
MS_time_rmanova = SSB_rmanova / DF_time_rmanova
MS_error_rmanova = SSW_rmanova / DF_error_rmanova

print("Mean squares:")
print(f" MSB (time) = {MS_time_rmanova:.4f}")
print(f" MSW (error) = {MS_error_rmanova:.4f}")

In [ ]:
# Calculate F-ratio and associated P value
f_ratio_rmanova = MS_time_rmanova / MS_error_rmanova
p_value_rmanova = f_dist(
    dfn=DF_time_rmanova,
    dfd=DF_error_rmanova).sf(f_ratio_rmanova)

print(f"With an F-ratio = {f_ratio_rmanova:.4f}, \
the associated P value = {p_value_rmanova:.5f}")

In [ ]:

# Calculate critical F-value
f_crit_rmanova = f_dist(
    dfn=DF_time_rmanova, dfd=DF_error_rmanova).ppf(1 - α)

# Generate x values for plotting
x_f_rmanova = np.linspace(0, 10, 500)
hx_f_rmanova = f_dist.pdf(
    x_f_rmanova, DF_time_rmanova, DF_error_rmanova)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x_f_rmanova, hx_f_rmanova, lw=2, color='black')

# Critical value
plt.axvline(
    x=f_crit_rmanova,
    color='orangered', linestyle='--',
    label=f"F* ({f_crit_rmanova:.2f})")

# Alpha area
plt.fill_between(
    x_f_rmanova[x_f_rmanova >= f_crit_rmanova],
    hx_f_rmanova[x_f_rmanova >= f_crit_rmanova],
    color='tomato',
    label=f"α ({α})")

# F-statistic
plt.axvline(
    x=f_ratio_rmanova,
    color='limegreen', linestyle='-.',
    label=f"F ({f_ratio_rmanova:.2f})")

# P value area
plt.fill_between(
    x_f_rmanova[x_f_rmanova >= f_ratio_rmanova],
    hx_f_rmanova[x_f_rmanova >= f_ratio_rmanova],
    color='greenyellow',
    label=f"P ({p_value_rmanova:.3f})")

plt.xlabel("F")
plt.ylabel('Density')
plt.ylim((0, .25))
plt.title(f"F-distribution ({DF_time_rmanova},{DF_error_rmanova})")
plt.margins(x=0.01, y=0)
plt.legend();

### Repeated measures ANOVA using Python
#### rmANOVA using Pingouin


In [ ]:
# Test for sphericity
spher_results = pg.sphericity(
    data=data_rmanova_long,
    dv='blood_pressure',
    within='time',
    subject='case')

print("Mauchly's test of sphericity:")
# Enhanced presention as a Series
print(pd.Series(spher_results._asdict()))

In [ ]:
# Perform rmANOVA with automatic sphericity correction
aov_rm = pg.rm_anova(
    data=data_rmanova_long,
    dv='blood_pressure',
    within='time',
    subject='case',
    detailed=True,
    effsize="ng2",     # Use generalized eta-squared as effect size
    correction='auto'  # Automatically apply sphericity correction if needed
)

print("Repeated-measures ANOVA (Pingouin):")
print(aov_rm.round(5))

#### Repeated measures ANOVA using statsmodels


In [ ]:
from statsmodels.stats.anova import AnovaRM

# Perform rmANOVA using AnovaRM
model_anova_rm_statsmodels = AnovaRM(
    data=data_rmanova_long,
    depvar='blood_pressure',
    subject='case',
    within=['time'])

results_anova_rm_statsmodels = model_anova_rm_statsmodels.fit()

# Print the ANOVA table
print("Repeated-measures ANOVA table (statsmodels):")
print(results_anova_rm_statsmodels.anova_table.round(5))

## Non-parametric methods for ANOVA and rmANOVA
### Kruskal-Wallis test (one-way ANOVA) 


#### Kruskal-Wallis test using Pingouin


In [ ]:
# Perform Kruskal-Wallis test using Pingouin
results_kruskal_pingouin = pg.kruskal(
    data=data_anova,
    dv='Pain threshold',
    between='Hair color')

print("Non-parametric Kruskal-Wallis one-way ANOVA (Pingouin):")
print(results_kruskal_pingouin.round(4))

#### Kruskal-Wallis test using SciPy


In [ ]:
from scipy.stats import kruskal

# Perform nonparametric one-way ANOVA using SciPy and 
# the group data extracted in the previous parametric test
F_kruskal_scipy, p_kruskal_scipy = kruskal(*groups)

print("Non-parametric Kruskal-Wallis one-way ANOVA (SciPy):")
print(f"F-statistic = {F_kruskal_scipy:.3f} with P value = \
{p_kruskal_scipy:.4f}")

### Friedman test (rmANOVA)


#### Friedman's test using Pingouin


In [ ]:
# Perform Friedman's test using Pingouin (χ² test method)
results_friedman_pingouin = pg.friedman(
    data=data_rmanova_long,
    dv='blood_pressure',
    within='time',
    subject='case')

print("Non-parametric Friedman rmANOVA (Pingouin; χ² test):")
print(results_friedman_pingouin.round(4))

In [ ]:
# Perform Friedman's test using Pingouin (F-test method)
results_friedman_pingouin = pg.friedman(
    data=data_rmanova_long,
    dv='blood_pressure',
    within='time',
    subject='case',
    method='f')

print("Non-parametric Friedman rmANOVA (Pingouin; F-test):")
print(results_friedman_pingouin.round(4))

#### Friedman's test using SciPy


In [ ]:
from scipy.stats import friedmanchisquare

# Extract the data for each time point as Series
before_data = data_rmanova['before']
during_data = data_rmanova['during']
after_data = data_rmanova['after']

# Perform Friedman's test using SciPy
Q_friedman_scipy, p_friedman_scipy = friedmanchisquare(
    before_data, during_data, after_data)

print("Non-parametric Friedman rmANOVA (SciPy):")
print(f"Q-statistic = {Q_friedman_scipy:.4f} with P value = \
{p_friedman_scipy:.4f}")

## Two-way ANOVA
### Example of two-way ANOVA


In [ ]:
# Load the 'anova2' dataset
data_two_way_anova = pg.read_dataset('anova2')

print("First rows of the fertilizer dataset:")
print(data_two_way_anova.head())

In [ ]:
print(
    data_two_way_anova
    .set_index(['Crop', 'Blend', 'Ss'])  # 'Ss' is the unique identifier
    .unstack(0))

In [ ]:

plt.figure(figsize=(3.5, 3))

# Create the boxplot
sns.boxplot(
    x='Crop',
    y='Yield',
    data=data_two_way_anova,
    fliersize=0,
    hue='Blend')

# Overlay the stripplot
sns.stripplot(
    x='Crop', y='Yield', data=data_two_way_anova,
    hue='Blend', dodge=True,  # Separate points based on blend
    palette='dark:black', legend=False)

plt.title('Interaction plot (two-way ANOVA)');

### How two-way ANOVA works


### Manual calculation of two-way ANOVA


In [ ]:
# Calculate the grand mean of 'Yield'
grand_mean_two_way_anova = data_two_way_anova['Yield'].mean()

# Calculate the number of observations per cell using pivot_table
n_cells_two_way_anova = data_two_way_anova.pivot_table(
    index='Crop', columns='Blend', values='Yield', aggfunc='count')

# Store the number of observations per cell
n_per_cell = n_cells_two_way_anova.mean(axis=None)

# Print all results
print("Grand mean for the two-way ANOVA =",
    round(grand_mean_two_way_anova,2))
print()
print("Number of observations per cell:")
print(n_cells_two_way_anova)
print()
print(f"Number of observations per cell (m) = {n_per_cell}")

In [ ]:
# Calculate mean and count of Yield for each *Crop*
data_two_way_anova_crop = (
    data_two_way_anova
    .groupby('Crop')['Yield']
    .agg(['mean', 'count']))

# Calculate SS_A (*Crop*), i.e., mean count per Crop
n_crop_two_way_anova = data_two_way_anova_crop['count'].mean()
SS_crop_two_way_anova = sum(
    count * (mean - grand_mean_two_way_anova)**2
    for mean, count in data_two_way_anova_crop.values)

# Number of levels and degrees of freedom for *Crop* factor
n_levels_crop_two_way_anova = data_two_way_anova['Crop'].nunique()
DF_crop_two_way_anova = n_levels_crop_two_way_anova - 1

# Print all results
print("Summary statistics for Yield by Crop:")
print(data_two_way_anova_crop.round(2))
print()
print(f"SS for Crop (SS_A) = {SS_crop_two_way_anova:.3f}")
print(f"Number of levels for Crop factor (r) = \
{n_levels_crop_two_way_anova}")
print(f"Degrees of freedom for Crop factor = {DF_crop_two_way_anova}")

In [ ]:
# Calculate mean and count of Yield for each *Blend*
data_two_way_anova_blend = (
    data_two_way_anova
    .groupby('Blend')['Yield']
    .agg(['mean', 'count']))

# Calculate SS_B (*Blend*), i.e., mean count per Blend
n_blend_two_way_anova = data_two_way_anova_blend['count'].mean()
SS_blend_two_way_anova = sum(
    count * (mean - grand_mean_two_way_anova)**2
    for mean, count in data_two_way_anova_blend.values)

# Number of levels and degrees of freedom for *Blend* factor
n_levels_blend_two_way_anova = data_two_way_anova['Blend'].nunique()
DF_blend_two_way_anova = n_levels_blend_two_way_anova - 1

# Print all results
print("Summary statistics for Yield by Blend:")
print(data_two_way_anova_blend.round(2))
print()
print(f"SS for Blend (SS_B) = {SS_blend_two_way_anova:.3f}")
print(f"Number of levels for Blend factor (c) = \
{n_levels_blend_two_way_anova}")
print(f"Degrees of freedom for Blend factor = {DF_blend_two_way_anova}")

In [ ]:
# Calculate the mean Yield for each combination of *Crop* and *Blend*
data_two_way_anova_interaction_mean = (
    data_two_way_anova
    .groupby(['Crop', 'Blend'])['Yield']
    .mean())

# Initialize the sum of squares for the interaction (SS_AxB)
SS_interaction_two_way_anova = 0

# Loop through each unique value of *Crop*
for crop in data_two_way_anova['Crop'].unique():
    # Loop through each unique value of *Blend*
    for blend in data_two_way_anova['Blend'].unique():
        # Get the mean Yield for the current combination of *Crop* / *Blend*
        mean_crop_two_way_anova_blend = (
            data_two_way_anova_interaction_mean.loc[(crop, blend)])
        # Get the mean Yield for the current *Crop*
        mean_crop_two_way_anova = data_two_way_anova_crop.loc[crop, 'mean']
        # Get the mean Yield for the current *Blend*
        mean_blend_two_way_anova=data_two_way_anova_blend.loc[blend,'mean']
        # Add the weighted squared deviation to SS_interaction_two_way_anova
        SS_interaction_two_way_anova += (
            n_per_cell * (
                mean_crop_two_way_anova_blend
                - mean_crop_two_way_anova
                - mean_blend_two_way_anova
                + grand_mean_two_way_anova)**2)

# Degrees of freedom for interaction
DF_interaction_two_way_anova = DF_crop_two_way_anova*DF_blend_two_way_anova

# Print all results
print("Summary statistics for Yield by Crop:Blend:")
print(data_two_way_anova_interaction_mean.round(2))
print()
print(f"SS for interaction (SS_AxB) = {SS_interaction_two_way_anova:.3f}")
print(f"Degrees of freedom for interaction = \
{DF_interaction_two_way_anova}")

In [ ]:
# SST (total sum of squares)
SST_two_way_anova = (
    (data_two_way_anova['Yield'] - grand_mean_two_way_anova)**2
).sum()

# Total number of samples and degrees of freedom for total variation
# n_total_two_way_anova = (
#     n_per_cell
#     * n_levels_crop_two_way_anova
#     * n_levels_blend_two_way_anova)
n_total_two_way_anova = len(data_two_way_anova)

DF_total_two_way_anova = n_total_two_way_anova - 1

# Print all results
print(f"SS total (SST) = {SST_two_way_anova:.3f}")
print(f"Total number of samples (n) = {n_total_two_way_anova}")
print(f"Degrees of freedom for total variation = {DF_total_two_way_anova}")

In [ ]:
# Calculate SSE from scratch
SSW_two_way_anova = (
    data_two_way_anova                  # Start with the DataFrame
    .groupby(['Crop', 'Blend'])         # Group the data by Crop and Blend
    .Yield                              # Select the 'Yield' column
    .apply(lambda x: (x - x.mean())**2) # Squared diff from group mean
    .sum()                              # Sum squared deviations → SSW
)

# Degrees of freedom for error/residuals
DF_within_two_way_anova = (
    n_total_two_way_anova 
    - n_levels_crop_two_way_anova 
    * n_levels_blend_two_way_anova)

print(f"SSW (calculated from scratch) = {SSW_two_way_anova:.3f}")
print(f"Degrees of freedom for error = {DF_within_two_way_anova}")

In [ ]:
# Calculate SSW by substracting other SS from SST
SSW_substraction_two_way_anova = (
    SST_two_way_anova 
    - SS_crop_two_way_anova 
    - SS_blend_two_way_anova 
    - SS_interaction_two_way_anova)

# Degrees of freedom for error from substraction
DF_within_substraction_two_way_anova = (
    DF_total_two_way_anova
    - DF_blend_two_way_anova
    - DF_crop_two_way_anova
    - DF_interaction_two_way_anova)

print(f"SSW (calculated by substraction) = \
{SSW_substraction_two_way_anova:.3f}")
print(f"Degrees of freedom for error (from substraction) = \
{DF_within_substraction_two_way_anova}")

In [ ]:
# Calculate MS
MS_crop_two_way_anova = SS_crop_two_way_anova / DF_crop_two_way_anova
MS_blend_two_way_anova = SS_blend_two_way_anova / DF_blend_two_way_anova
MS_interaction_two_way_anova = (
    SS_interaction_two_way_anova / DF_interaction_two_way_anova)
MS_within_two_way_anova = SSW_two_way_anova / DF_within_two_way_anova

# Calculate each F-ratio
F_crop_two_way_anova = MS_crop_two_way_anova / MS_within_two_way_anova
F_blend_two_way_anova = MS_blend_two_way_anova / MS_within_two_way_anova
F_interaction_two_way_anova = (
    MS_interaction_two_way_anova / MS_within_two_way_anova)

# Print all results
print(f"MS for Crop factor = {MS_crop_two_way_anova:.3f}")
print(f"MS for Blend factor = {MS_blend_two_way_anova:.3f}")
print(f"MS for interaction = {MS_interaction_two_way_anova:.3f}")
print(f"MS within = {MS_within_two_way_anova:.3f}")
print()
print(f"F-ratio for Crop factor = {F_crop_two_way_anova:.3f}")
print(f"F-ratio for Blend factor = {F_blend_two_way_anova:.3f}")
print(f"F-ratio for interaction = {F_interaction_two_way_anova:.3f}")

In [ ]:
# Calculate the P values
p_val_crop_two_way_anova = f_dist.sf(
    F_crop_two_way_anova, DF_crop_two_way_anova, DF_within_two_way_anova)
p_val_blend_two_way_anova = f_dist.sf(
    F_blend_two_way_anova, DF_blend_two_way_anova, DF_within_two_way_anova)
p_val_interaction_two_way_anova = f_dist.sf(
    F_interaction_two_way_anova,
    DF_interaction_two_way_anova,
    DF_within_two_way_anova)

print(f"P value for Crop factor = {p_val_crop_two_way_anova:.4f}")
print(f"P value for Blend factor = {p_val_blend_two_way_anova:.4f}")
print(f"P value for interaction = {p_val_interaction_two_way_anova:.4f}")

In [ ]:

# Function to plot F-distribution with critical values and P value
def plot_f_distribution(title, dfn, f_ratio, p_value, ax):
    f_crit = f_dist(dfn=dfn, dfd=DF_within_two_way_anova).ppf(1 - α)
    # dfd is always DF_within_two_way_anova
    x_f = np.linspace(0, 10, 200)
    hx_f = f_dist.pdf(x_f, dfn, DF_within_two_way_anova)
    ax.plot(x_f, hx_f, lw=2, color='black')
    ax.axvline(
        x=f_crit, color='orangered', linestyle='--',
        label=f"F* ({f_crit:.2f})")
    ax.fill_between(
        x_f[x_f >= f_crit], hx_f[x_f >= f_crit], color='tomato',
        label=f"α ({α})")
    ax.axvline(
        x=f_ratio, color='limegreen', linestyle='-.',
        label=f"F ({f_ratio:.2f})")
    ax.fill_between(
        x_f[x_f >= f_ratio], hx_f[x_f >= f_ratio], color='greenyellow',
        label=f"P ({p_value:.3f})")
    ax.set_xlabel("F")
    ax.set_ylabel('')
    ax.set_title(f"{title} ({dfn},{DF_within_two_way_anova})")
    ax.margins(x=0.025, y=0)
    ax.legend()

# List of parameters
F_plot_crop = ("Crop", DF_crop_two_way_anova,
    F_crop_two_way_anova, p_val_crop_two_way_anova)
F_plot_blend = ("Blend", DF_blend_two_way_anova,
    F_blend_two_way_anova, p_val_blend_two_way_anova)
F_plot_interaction = ("Interaction", DF_interaction_two_way_anova,
    F_interaction_two_way_anova, p_val_interaction_two_way_anova)

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(6.5, 2.5))

# Loop through sources of variation and plot F-distributions
for i, (title, df, f_ratio, p_value) in enumerate(
    (F_plot_crop, F_plot_blend, F_plot_interaction)):
    plot_f_distribution(title, df, f_ratio, p_value, axes[i])

plt.tight_layout();

### Two-way ANOVA using Python
#### Two-way ANOVA with Pingouin


In [ ]:
# Perform two-way ANOVA using Pingouin
aov2 = data_two_way_anova.anova(
    dv='Yield',
    between=['Crop', 'Blend'])

print("Two-way ANOVA (Pinguin):")
print(aov2.round(4))

#### Two-way ANOVA with statsmodels


In [ ]:
formula_two_way_anova = "Yield ~ C(Crop)*C(Blend)"
#formula_two_way_anova = "Yield ~ C(Crop) + C(Blend) + C(Crop):C(Blend)"

# Prepare the one-way ANOVA model using statsmodels
model_two_way_anova_statsmodels = ols(
    formula=formula_two_way_anova,
    data=data_two_way_anova)

# Fit the model
results_two_way_anova_statsmodels = model_two_way_anova_statsmodels.fit()

# Print the ANOVA table
print(anova_lm(results_two_way_anova_statsmodels).round(4))

### Unbalanced design in two-way ANOVA


In [ ]:
# Load the 'anova2_unbalanced' dataset
data_unbalanced = pg.read_dataset('anova2_unbalanced')
print("First rows of the diet dataset:")
print(data_unbalanced.head())

In [ ]:
# Calculate the number of observations per cell using pivot_table
print("Number of observations per cell:")
print(
    data_unbalanced.pivot_table(
        index='Diet',
        columns='Exercise',
        values='Scores',
        aggfunc='count'))

In [ ]:

plt.figure(figsize=(3.5, 3))

# Create the boxplot
sns.boxplot(
    x='Diet', y='Scores', data=data_unbalanced,
    hue='Exercise')

# Overlay the stripplot
sns.stripplot(
    x='Diet', y='Scores', data=data_unbalanced,
    hue='Exercise', dodge=True,
    palette='dark:black',legend=False)

plt.title("Interaction plot (unbalanced)");

In [ ]:
# Perform two-way ANOVA
aov2_unbalanced = data_unbalanced.anova(
    dv="Scores",
    between=["Diet", "Exercise"],
    detailed=True,  # default for N-way ANOVA
    ss_type=2,      # default
)

print("Unbalanced two-way ANOVA (Pingouin):")
print(aov2_unbalanced.round(4))

In [ ]:
results_unbalanced_anova_statsmodels = ols(
    formula="Scores ~ Diet*Exercise",
    # Diet and Exercise are of type string ~ readily categories
    data=data_unbalanced
).fit()

# Print the ANOVA table with type I sums of squares
print("Unbalanced two-way ANOVA table (statsmodels; type I):")
print(
    anova_lm(results_unbalanced_anova_statsmodels, typ=1).round(4))

In [ ]:
# Print the ANOVA table with type II sums of squares
print("Unbalanced two-way ANOVA table (statsmodels; type II):")
print(
    anova_lm(results_unbalanced_anova_statsmodels, typ=2).round(4))

In [ ]:
# Print the ANOVA table with type III sums of squares
print("Unbalanced two-way ANOVA table (statsmodels; type III):")
print(
    anova_lm(results_unbalanced_anova_statsmodels, typ=3).round(4))

## Special designs


### Mixed-design ANOVA


In [ ]:
# Load the 'mixed_anova' dataset
data_mixed = pg.read_dataset('mixed_anova')

# Display the first few rows of the DataFrame
print("First rows of the memory dataset:")
print(data_mixed.head())

In [ ]:

plt.figure(figsize=(3.5, 3))

# Create the boxplot
sns.boxplot(
    x='Time',
    y='Scores',
    data=data_mixed,
    order=['January', 'June', 'August'],  # Alphabetical by default
    hue='Group',
    fill=False,
    gap=.1)

# Add plot labels and title
plt.title('Mixed-design ANOVA example');

In [ ]:
print("Data for Subject No. 0:")
print(data_mixed.query('Subject == 0'))

In [ ]:

# Get unique values of *Group*
groups_mixed_anova = data_mixed['Group'].unique()
n_groups_mixed_anova = len(groups_mixed_anova)

# Create subplots
fig, axes = plt.subplots(
    nrows=1,
    ncols=n_groups_mixed_anova,
    figsize=(6, 6/n_groups_mixed_anova),
    sharey=True)

# Loop through groups and create paired plots
for i, group in enumerate(groups_mixed_anova):
    pg.plot_paired(
        data=data_mixed[data_mixed['Group'] == group],
        dv='Scores',
        within='Time',
        order=['January', 'June', 'August'],  # Alphabetical by default
        subject='Subject',
        boxplot=True,
        orient='v',
        ax=axes[i],  # Assign plot to the correct subplot
        boxplot_kwargs={'color': 'white', 'linewidth': 2, 'zorder': 1})
    axes[i].set_title(group)
    axes[i].set_xlabel('')
    sns.despine(trim=True, ax=axes[i])

plt.tight_layout();

In [ ]:
# Perform mixed-design ANOVA
aov_mixed = pg.mixed_anova(
    data=data_mixed,
    dv='Scores',
    between='Group',
    within='Time',
    subject='Subject',)

print("Mixed-design ANOVA (Pingouin):")
print(aov_mixed.round(3))

### N-way ANOVA


In [ ]:
# Load the 'anova3' dataset
data_three_way_anova = pg.read_dataset('anova3')

# Display the first few rows of the DataFrame
print("First rows of the three-way cholesterol dataset:")
print(data_three_way_anova.head())

In [ ]:

# Create the catplot with 'Drug' as the panel separation
g = sns.catplot(
    x='Drug',
    y='Cholesterol',
    hue='Sex',
    col='Risk',  # Separate panels by 'Risk'
    data=data_three_way_anova,
    kind='box',
    dodge=True,  # Separate points for different 'Dosage' levels
    height=2.5,
    fill=False)

# Add plot title
g.figure.subplots_adjust(top=.8)
g.figure.suptitle(
    'Three-way ANOVA: Cholesterol by Drug, Sex, and Risk');

In [ ]:
print("Count per combination of Drug, Sex and Risk:")
print(
    data_three_way_anova
    .groupby(['Drug', 'Sex', 'Risk'])
    .agg(['count']))

In [ ]:
# Perform three-way ANOVA with Pingouin
aov3 = data_three_way_anova.anova(
    dv='Cholesterol',
    between=['Drug', 'Sex', 'Risk'],
)

print("Threes-way ANOVA (Pingouin):")
print(aov3.round(3))

In [ ]:
# Perform three-way ANOVA with statsmodels
results_three_way_anova_statsmodels = ols(
    formula="Cholesterol ~ Drug * Sex * Risk",
    # Notice that Drug, Sex, and Risk are of type string (readily categories)
    # formula="""
    # Cholesterol ~ Drug + Sex + Risk +
    # Drug:Sex + Drug:Risk + Sex:Risk +
    # Drug:Sex:Risk
    # """,
    data=data_three_way_anova
).fit()

# Print the ANOVA table
print("Three-way ANOVA table (statsmodels):")
print(anova_lm(results_three_way_anova_statsmodels).round(3))

In [ ]:
# Load the 'anova3_unbalanced' dataset
data_unbalanced_three_way_anova = pg.read_dataset('anova3_unbalanced')

print("Count per combination of drug, gender and risk:")
print(
    data_unbalanced_three_way_anova
    .groupby(['Drug', 'Sex', 'Risk'])
    .agg(['count']))

In [ ]:
# Unbalanced 3-way ANOVA with Pingouin
aov3_unbalanced = data_unbalanced_three_way_anova.anova(
    dv='Cholesterol',
    between=['Drug', 'Sex', 'Risk'],
    ss_type=2)

print("Unbalanced 3-way ANOVA (Pingouin):")
print(aov3_unbalanced.round(3))

### Two-way repeated measures ANOVA


In [ ]:
# Load the 'rm_anova2' dataset
data_two_way_rmanova = pg.read_dataset('rm_anova2')

print("Sample rows of the employee performance dataset:")
print(data_two_way_rmanova.sample(10))

In [ ]:
# Print the number of unique Subjects
print(f"There are {data_two_way_rmanova['Subject'].nunique()} \
unique Subjects")

# Pivot to the mean of the Performance values calculated 
# for each combination of Time and Metric
print()
print("Pivot table for each combination of Time and Metric:")
print(
    data_two_way_rmanova.pivot_table(
        index='Time',
        columns='Metric',
        values='Performance',
        aggfunc=['count']))

In [ ]:
# Display the data for Subject #1
print("Performance evaluation for Subject No. 1:")
print(data_two_way_rmanova.query('Subject == 1'))

In [ ]:

# Create subplots - one for each metric
metrics = data_two_way_rmanova['Metric'].unique()[::-1]
n_metrics = len(metrics)

fig, axes = plt.subplots(
    nrows=1,
    ncols=n_metrics,
    figsize=(6, 6/n_metrics),
    sharey=True)

# Loop through metrics and create paired plots
for i, metric in enumerate(metrics):
    pg.plot_paired(
        data=data_two_way_rmanova[data_two_way_rmanova['Metric'] == metric],
        dv='Performance',
        within='Time',
        order=['Pre', 'Post'],
        subject='Subject',
        boxplot=True,
        orient='v',
        ax=axes[i],
        boxplot_kwargs={'color': 'white', 'linewidth': 2, 'zorder': 1})
    axes[i].set_title(metric)
    axes[i].set_xlabel('')
    sns.despine(trim=True, ax=axes[i])

plt.tight_layout();

In [ ]:
warnings.simplefilter(action='ignore', category=FutureWarning)

# Two-way repeated measures ANOVA with Pingouin
aov2_rm = data_two_way_rmanova.rm_anova(
    dv='Performance',
    within=['Metric', 'Time'],
    subject='Subject')

print("Two-way rmANOVA (Pingouin):")
print(aov2_rm.round(4))

In [ ]:
# Perform two-way rmANOVA using AnovaRM
model_two_way_anova_rm_statsmodels = AnovaRM(
    data=data_two_way_rmanova,
    depvar='Performance',
    within=['Metric', 'Time'],
    subject='Subject',)

results_two_way_anova_rm_statsmodels=model_two_way_anova_rm_statsmodels.fit()
results_two_way_anova_rm_table_statsmodels = \
results_two_way_anova_rm_statsmodels.anova_table

# Print the ANOVA table
print("Two-way repeated-measures ANOVA table (statsmodels):")
print(results_two_way_anova_rm_table_statsmodels.round(6))

### Analysis of covariance (ANCOVA)


In [ ]:
# Load the 'ancova' dataset
data_ancova = pg.read_dataset('ancova')

print("First rows of the teaching method dataset:")
print(data_ancova.head())

In [ ]:
print("Normality test in ANCOVA:")
print(
    pg.normality(data=data_ancova, dv='Scores', group='Method'))

In [ ]:
print("Test for equal variance in ANCOVA:")
print(
    pg.homoscedasticity(data=data_ancova, dv='Scores', group='Method'))

In [ ]:

# Create the line plot
sns.lmplot(
    x='Income', y='Scores', data=data_ancova,
    hue='Method', ci=None, height=3)

# Add plot title and labels
plt.title('ANCOVA assumptions');

In [ ]:
# Perform ANCOVA using Pingouin
ancova_pingouin= pg.ancova(
    data=data_ancova,
    dv='Scores',
    between='Method',
    covar=['Income'],  # This can be a list of covariates
)

print("ANCOVA (Pingouin):")
print(ancova_pingouin.round(5))

### Repeated-measures correlation


In [ ]:
print("Repeated measures correlation (Pingouin):")
print(
    pg.rm_corr(
        data=data_ancova,
        x='Income',
        y='Scores',
        subject='Method').round(3))

In [ ]:

# Create the repeated measures correlation plot
pg.plot_rm_corr(
    data=data_ancova,
    x='Income',
    y='Scores',
    subject='Method',
    kwargs_facetgrid={'height': 3},
    legend=True)
plt.title("Repeated measures correlation");

## Multiple comparison tests after ANOVA


### Tukey HSD test


#### Calculating confidence intervals for mean differences
#### Tukey-Kramer test for unequal sample sizes


#### Tukey multiple comparison with Python
##### Tukey HSD with statsmodels


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Perform Tukey HSD test with statsmodels
tukeyhsd = pairwise_tukeyhsd(
    endog=data_anova['Pain threshold'],  # Dependent variable
    groups=data_anova['Hair color'],     # Grouping variable
    alpha=0.05,                          # Default
)

# Print the results
print(tukeyhsd.summary())

# The following works similarly; see Bonferroni's adjustment later
# from statsmodels.stats.multicomp import MultiComparison

# mult_comp = MultiComparison(
#     data_anova['Pain threshold'], data_anova['Hair color'])
# results_tukeyhsd = mult_comp.tukeyhsd()
# print(results_tukeyhsd.summary())

In [ ]:
# Print the critical q value
print(
    "Critical value of the studentized range statistic (q*) =",
    round(tukeyhsd.q_crit, 2))

In [ ]:

# Plot the results of the Tukey HSD test
tukeyhsd.plot_simultaneous(
    comparison_name='Light Blond',  # Reference
    xlabel='Pain threshold',
    ylabel='Hair color',
    figsize=(3.5, 3));

##### Tukey HSD with Pingouin


In [ ]:
# Perform Tukey HSD test using Pingouin
tukey_results_pingouin = pg.pairwise_tukey(
    data_anova,
    dv='Pain threshold',
    between='Hair color',)

print("Tukey HSD test (Pingouin):")
print(tukey_results_pingouin.round(4))

##### Tukey HSD with scikit-posthocs


In [ ]:
import scikit_posthocs as ph

# Perform Tukey HSD test using scikit-posthocs
tukey_results_ph = ph.posthoc_tukey_hsd(
    data_anova,
    val_col='Pain threshold',
    group_col='Hair color',
    sort=True,  # Sort by `group_col`
)

print("Tukey HSD (scikit-posthocs; adjusted P values):")
print(
    tukey_results_ph.round(4))

In [ ]:
# Create a significance table from the previous Tukey HSD results
print("Tukey HSD (scikit-posthocs; significance table):")
print(
    ph.sign_table(tukey_results_ph))

In [ ]:

# Create a significance plot from the Tukey HSD results
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))

ph.sign_plot(
    tukey_results_ph,
    cmap=['1', '#fb6a4a', '#08306b', '#4292c6', '#c6dbef'], # Custom colors
    linewidths=0.5,                        # Adjust linewidths
    linecolor='0',                         # Line color
    clip_on=False,                         # Prevent clipping
    square=True,                           # Make cells square
    cbar_ax_bbox= [0.85, 0.35, 0.05, 0.3], # Colorbar position
    ax=ax)

# Add plot title
plt.suptitle("Significance plot (Tukey HSD)")
plt.tight_layout();

### Dunnett's test


In [ ]:
# Dunnett's critical value
t_dunnett_crit = 2.610
print(f"Critical Dunnett (α=0.05, DF={DF_within}, r={r}) t* = \
{t_dunnett_crit:.3f}")

control_dunnett_label = 'Light Blond'

print()
print(f"Dunnett's test with reference = {control_dunnett_label}:")

# Exclude control group label
groups_dunnett_label = [
    color for color in hair_colors if color != control_dunnett_label]

# Compare each group to *Light Blond* using the previous summary statistics
control_mean = group_statistics.loc[control_dunnett_label, 'mean']
control_count = group_statistics.loc[control_dunnett_label, 'count']

for group in groups_dunnett_label:
    group_mean = group_statistics.loc[group, 'mean']
    group_count = group_statistics.loc[group, 'count']

    t_dunnett_stat = (
        (group_mean - control_mean)
        / np.sqrt(MS_within * (1/group_count + 1/control_count)))

    print(f"*{group} vs. Light Blond")
    print(f" Dunnett t-statistic =", round(t_dunnett_stat, 3))
    print(f" Significant? {abs(t_dunnett_stat) > t_dunnett_crit}")

In [ ]:
from scipy.stats import dunnett

# Extract the data for each hair color group
groups_dunnett = [
    data_anova['Pain threshold'][data_anova['Hair color'] == color]
    for color in groups_dunnett_label]

control_dunnett = data_anova.loc[
    data_anova['Hair color'] == control_dunnett_label, 'Pain threshold']

# Perform Dunnett's test using SciPy
res = dunnett(*groups_dunnett, control=control_dunnett)

# Print the results
print("Dunnett's test results (SciPy):")
for i, group in enumerate(groups_dunnett_label):
    print(f"{group} vs. {control_dunnett_label}:\
\n statistic = {res.statistic[i]:.3f}; P value = {res.pvalue[i]:.4f}")

In [ ]:
# Perform Dunnett's test using scikit-posthocs
dunnett_results_ph = ph.posthoc_dunnett(
    data_anova,
    val_col='Pain threshold',
    group_col='Hair color',
    control='Light Blond',
    to_matrix=False,  # We don't need a full matrix of P values in Dunnett
)

print("Dunnett's test (scikit-posthocs; adjusted P values):")
print(dunnett_results_ph.round(3))

In [ ]:
# Create a significance table from the Dunnett's test results
print("Dunnett's test (scikit-posthocs; significance array):")
print(
    ph.sign_table(dunnett_results_ph))

### Bonferroni's adjustment


In [ ]:
from statsmodels.stats.multicomp import MultiComparison
from scipy.stats import ttest_ind

# Create a MultiComparison object
mod = MultiComparison(
    data=data_anova['Pain threshold'],
    groups=data_anova['Hair color'],
    group_order=[
    'Light Blond', 'Dark Blond', 'Light Brunette', 'Dark Brunette'])

# Perform the t-tests with Bonferroni adjustment
results_bonferroni = mod.allpairtest(testfunc=ttest_ind, method='bonf')

# Print the results
print(results_bonferroni[0])

In [ ]:
# Multiple t-tests with correction using Pingouin
post_hocs_pingouin = pg.pairwise_tests(
    dv='Pain threshold',
    between='Hair color',
    data=data_anova,
    correction=False,  # Correction for unequal variances
    padjust='bonf')

print("Multiple t-tests with Bonferroni adjustment (Pingouin):")
print(post_hocs_pingouin.round(4))

In [ ]:
# Perform t-tests with Bonferroni adjustment using scikit-posthocs
bonferroni_results_ph = ph.posthoc_ttest(
    data_anova,
    val_col='Pain threshold',
    group_col='Hair color',
    sort=True,  # Sort by `group_col`
    pool_sd=False,
    p_adjust='bonferroni')

print("Multiple t-tests (scikit-posthocs; Bonferroni adjusted P values):")
print(bonferroni_results_ph)

In [ ]:
# Create a significance table from the Bonferroni's test results
print("Multiple t-tests (scikit-posthocs; Bonferroni significance table):")
print(
    ph.sign_table(
        bonferroni_results_ph,
        # Show data above the diagonal only
        lower=False))

In [ ]:

# Create a significance plot from the Tukey HSD results
_, ax = plt.subplots(nrows=1, ncols=1, figsize=(3.5, 3))

ph.sign_plot(
    bonferroni_results_ph,
    cmap=['1', '#fb6a4a', '#08306b', '#4292c6', '#c6dbef'],
    linewidths=0.5, linecolor='0',
    clip_on=False, square=True,
    cbar_ax_bbox= [0.85, 0.35, 0.05, 0.3], ax=ax)

# Add plot title
plt.suptitle("Significance plot (Bonferroni)")
plt.tight_layout();

In [ ]:
# Specify the pairs we're interested in
pairs = [
    ('Dark Blond', 'Light Blond'),
    ('Dark Brunette', 'Light Brunette'),
    ('Light Blond', 'Light Brunette')]

# Loop over the pairs, perform t-tests, and collect P values
p_values_multitests = []
for pair in pairs:
    group1=data_anova['Pain threshold'][data_anova['Hair color']==pair[0]]
    group2=data_anova['Pain threshold'][data_anova['Hair color']==pair[1]]
    t_statistic, p_value = ttest_ind(group1, group2)
    p_values_multitests.append(p_value)

# Apply Bonferroni adjustment using pingouin.multicomp
_, adjusted_p_values_bonf = pg.multicomp(p_values_multitests, method='bonf')

# Print the adjusted P values
print("Adjusted P values (Bonferroni) for selected pairs \
(Pingouin `multicomp`):")
for pair, p_value in zip(pairs, adjusted_p_values_bonf):
    print(f" {pair}: {p_value:.3f}")

In [ ]:
# Adjust P values using Bonferroni adjustment
n_comparisons = len(p_values_multitests)  # Number of comparisons
# Multiply each P value by the number of comparisons
adjusted_p_values_manual = np.array(p_values_multitests) * n_comparisons

# Print the adjusted P values
print("Adjusted P values (Bonferroni) for selected pairs (manually):")
for pair, p_value in zip(pairs, adjusted_p_values_manual):
    print(f" {pair}: {p_value:.3f}")

### Other multiple comparison adjustments


In [ ]:
from statsmodels.stats.multitest import multipletests

# Apply Holm adjustment on P values using statsmodels multipletests
_, adjusted_p_values_holm, *_ = multipletests(
    p_values_multitests, method='holm')

# Print the adjusted P values
print("Adjusted P values (Holm) for selected pairs (`multipletests`):")
for pair, p_value in zip(pairs, adjusted_p_values_holm):
    print(f" {pair}: {p_value:.3f}")

In [ ]:
# Apply Holm-Sidak adjustment on P values
_, adjusted_p_values_holm_sidak, *_ = multipletests(
    p_values_multitests, method='holm-sidak')

# Print the adjusted P values
print("Adjusted P values (Holm-Sidak) for selected pairs \
(`multipletests`):")
for pair, p_value in zip(pairs, adjusted_p_values_holm_sidak):
    print(f" {pair}: {p_value:.3f}")

In [ ]:
# Apply FDR BH adjustment
_, adjusted_p_values_bh, *_ = multipletests(
    p_values_multitests, method='fdr_bh')

# Print the adjusted P values
print("Adjusted P values (FDR BH) for selected pairs (`multipletests`):")
for pair, p_value in zip(pairs, adjusted_p_values_bh):
    print(f" {pair}: {p_value:.3f}")

In [ ]:
# Perform t-tests with FDR BH adjustment using scikit-posthocs
bonferroni_results_ph = ph.posthoc_ttest(
    data_anova,
    val_col='Pain threshold',
    group_col='Hair color',
    sort=True,  # Sort by `group_col`
    pool_sd=False,
    p_adjust='fdr_bh')

print("Adjusted P values (FDR BH) for all pairs (scikit-posthoc):")
print(bonferroni_results_ph)

In [ ]:
# Perform t-tests with Scheffé adjustment using scikit-posthocs
scheffe_results_ph = ph.posthoc_scheffe(
    data_anova,
    val_col='Pain threshold',
    group_col='Hair color')


print("Multiple t-tests (scikit-posthocs; Scheffé adjusted P values):")
print(scheffe_results_ph)

## Conclusion
